In [0]:
#Checking the files are loaded into the Volumes
Files = dbutils.fs.ls("/Volumes/rag/bronze/raw_pdf/")

for F in Files:
    print(F.name)

In [0]:
# pipeline_config.py
RAW_DATA_PATH = "/Volumes/rag/bronze/raw_pdf/"
CHECKPOINT_PATH = "/Volumes/rag/bronze/checkpoints/"
BRONZE_TABLE = "rag.bronze.pdf_documents"

In [0]:
# imports
from pyspark.sql import functions as F

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS rag.bronze.checkpoints")

In [0]:
# ingesting into the bronze layer from Volumes

spark.sql("create schema if not exists rag.bronze")

bronze_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "binaryFile")
        .option("cloudFiles.schemaLocation", CHECKPOINT_PATH + "bronze_schema")
        .option("pathGlobFilter", "*.pdf")
        .load(RAW_DATA_PATH)
        .select(
            F.col("path").alias("file_path"),
            F.col("content").alias("raw_content"),
            F.col("length").alias("file_size"),
            F.current_timestamp().alias("ingestion_ts"),
            F.lit("pdf").alias("source_format")
        )
)

(
    bronze_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT_PATH + "bronze")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE)
)

In [0]:
metadata_df = spark.read.table("rag.bronze.pdf_documents")
metadata_df.show()